In [1]:
import sqlite3
import pandas as pd
import numpy as np
import hashlib
import os
from tqdm.auto import tqdm
from multiprocessing import Pool, cpu_count
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
from sklearn.feature_selection import VarianceThreshold

# Activar barras de progreso en pandas
tqdm.pandas()

# 1. CONEXIÓN Y EXTRACCIÓN
ruta_db = '/home/pedro/chembl_local/chembl_36/chembl_36_sqlite/chembl_36.db'
print("Bloque 1: Conectando a ChEMBL local...")

con = sqlite3.connect(ruta_db)
query = """
SELECT 
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.canonical_smiles,
    act.standard_type,
    act.standard_value,
    act.standard_units,
    act.standard_relation,
    act.pchembl_value,
    act.data_validity_comment,
    docs.chembl_id AS document_chembl_id,
    ass.description AS assay_description,
    ass.assay_type,
    ass.assay_organism,
    ass.assay_category,
    ass.assay_tax_id,
    ass.assay_strain,
    ass.assay_tissue,
    ass.assay_cell_type,
    ass.assay_subcellular_fraction,
    ass.bao_format,
    ass.variant_id
FROM activities act
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN compound_structures cs ON md.molregno = cs.molregno
LEFT JOIN docs ON act.doc_id = docs.doc_id
WHERE td.organism = 'Trypanosoma cruzi';
"""

df_master = pd.read_sql_query(query, con)
con.close()
print(f"Extracción completada: {len(df_master)} registros crudos.")

# 2. ONTOLOGÍA DE BLANCOS TERAPÉUTICOS
diccionario_blancos = {
    'CHEMBL368': 'T_cruzi_General',
    'CHEMBL5131': 'Tripanotion_reductasa',
    'CHEMBL3563': 'Cruzaina',
    'CHEMBL57110': 'CYP51',
    'CHEMBL1075110': 'CYP51',
    'CHEMBL4105902': 'Trans_sialidasa',
    'CHEMBL5126': 'Trans_sialidasa'
}
df_master['blanco_terapeutico'] = df_master['target_chembl_id'].map(diccionario_blancos)

Bloque 1: Conectando a ChEMBL local...
Extracción completada: 145656 registros crudos.


In [2]:
print("Bloque 2: Curación de datos...")

# --- RUTA A: CLASIFICACIÓN (Percent Effect > 70%) ---
df_percent = df_master[df_master['standard_type'] == 'Percent Effect'].copy()
df_percent['standard_value'] = pd.to_numeric(df_percent['standard_value'], errors='coerce')
df_percent = df_percent.dropna(subset=['standard_value', 'canonical_smiles', 'blanco_terapeutico'])

df_percent['actividad_binaria'] = (df_percent['standard_value'] > 70.0).astype(int)
df_percent_curado = df_percent.groupby(['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico'])['actividad_binaria'].median().reset_index()
df_percent_curado['actividad_binaria'] = df_percent_curado['actividad_binaria'].round().astype(int)
print(f"Set de Clasificación listo: {len(df_percent_curado)} compuestos.")

# --- RUTA B: CURACIÓN MÁXIMA DE IC50 ---
df_ic50 = df_master[
    (df_master['standard_type'] == 'IC50') & 
    (df_master['standard_relation'] == '=') & 
    (df_master['standard_units'] == 'nM')
].copy()

df_ic50['standard_value'] = pd.to_numeric(df_ic50['standard_value'], errors='coerce')
df_ic50 = df_ic50[df_ic50['standard_value'] > 0]

def calc_pchembl(row):
    if pd.isna(row.get('pchembl_value')) and pd.notna(row['standard_value']):
        return 9.0 - np.log10(row['standard_value'])
    return row.get('pchembl_value')

df_ic50['pchembl_value'] = df_ic50.apply(calc_pchembl, axis=1)
df_ic50 = df_ic50.dropna(subset=['pchembl_value', 'canonical_smiles', 'blanco_terapeutico'])

# Filtro de mutantes
mutant_keywords = ['mutant', 'mutation', 'variant']
pattern = '|'.join(mutant_keywords)
is_mutant_text = df_ic50['assay_description'].str.contains(pattern, case=False, na=False)
no_variant_id = df_ic50['variant_id'].isna()
df_ic50 = df_ic50[~(is_mutant_text & no_variant_id)]

# Purga de discrepancias matemáticas
duplicados_mask = df_ic50.duplicated(subset=['molecule_chembl_id'], keep=False)
df_unicos = df_ic50[~duplicados_mask].copy()
df_multiples = df_ic50[duplicados_mask].copy()

def purgar_actividad(grupo):
    valores = grupo['pchembl_value'].values
    indices_descartar = set()
    for i in range(len(valores)):
        for j in range(i + 1, len(valores)):
            diff = abs(valores[i] - valores[j])
            if np.isclose(diff, 0.0, atol=1e-5) or np.isclose(diff, 3.0, atol=1e-5):
                indices_descartar.add(grupo.index[i])
                indices_descartar.add(grupo.index[j])
    return grupo.drop(list(indices_descartar))

if not df_multiples.empty:
    df_multiples_limpio = df_multiples.groupby('molecule_chembl_id', group_keys=False).apply(purgar_actividad)
else:
    df_multiples_limpio = pd.DataFrame()

df_ic50_curado = pd.concat([df_unicos, df_multiples_limpio]).reset_index(drop=True)
df_ic50_curado = df_ic50_curado.groupby(['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico'])['pchembl_value'].mean().reset_index()

# Guardar reserva de IC50
df_ic50_curado.to_csv('t_cruzi_ic50_curado_puro.csv', index=False)
print(f"Set de IC50 Curado guardado: {len(df_ic50_curado)} compuestos.")

Bloque 2: Curación de datos...
Set de Clasificación listo: 68283 compuestos.
Set de IC50 Curado guardado: 7362 compuestos.


In [10]:
import pandas as pd
import numpy as np
from Bio import Align
from itertools import combinations
import os

print("BLOQUE 1: Preparación Biológica (Matriz IBMTL)")

# 1. Cargar el dataset de IC50 rigurosamente curado
df = pd.read_csv('t_cruzi_ic50_curado_puro.csv')
print(f"Moléculas curadas cargadas: {len(df)}")

# 2. Diccionario de Secuencias FASTA (UniProt IDs de T. cruzi)
# NOTA: He puesto fragmentos cortos para que el código corra rápido en tu prueba. 
# Debes reemplazar estos strings por las secuencias FASTA completas de UniProt.
secuencias_proteinas = {
    'Cruzaina': 'MRLVVVVVVVLLA... (P25779)', 
    'CYP51': 'MLLQSAALLG... (Q7Z1V1)',
    'Trans_sialidasa': 'MKALLVAL... (Q270L1)',
    'Tripanotion_reductasa': 'MSRQ... (P26221)',
    'T_cruzi_General': 'FENOTIPICO' # Datos celulares generales, no tienen secuencia
}

# 3. Calcular Matriz de Similitud Evolutiva (Alineamiento Global)
print("Calculando similitud evolutiva de secuencias (AA-GSS)...")
aligner = Align.PairwiseAligner()
aligner.mode = 'global'

matriz_similitud = {}
blancos = list(secuencias_proteinas.keys())

for b1 in blancos:
    matriz_similitud[b1] = {}
    for b2 in blancos:
        if b1 == 'T_cruzi_General' or b2 == 'T_cruzi_General':
            # Si es un ensayo celular general, la similitud estructural con una enzima aislada es 0
            matriz_similitud[b1][b2] = 0.0
        elif b1 == b2:
            matriz_similitud[b1][b2] = 1.0 # 100% de similitud consigo misma
        else:
            # Alineamiento entre proteínas distintas
            seq1 = secuencias_proteinas[b1]
            seq2 = secuencias_proteinas[b2]
            score = aligner.score(seq1, seq2)
            # Normalizar el score a porcentaje (0 a 1)
            max_score = max(len(seq1), len(seq2)) 
            similitud = score / max_score
            matriz_similitud[b1][b2] = round(similitud, 4)

print("Matriz de similitud calculada con éxito.")

BLOQUE 1: Preparación Biológica (Matriz IBMTL)
Moléculas curadas cargadas: 7362
Calculando similitud evolutiva de secuencias (AA-GSS)...
Matriz de similitud calculada con éxito.


In [11]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit.Chem import AllChem
from multiprocessing import Pool, cpu_count
from tqdm.auto import tqdm
from rdkit import RDLogger

print("\nBLOQUE 2: Generación de Fingerprints (MACCS + ECFP4)")
RDLogger.DisableLog('rdApp.*') # Silenciar RDKit

def calcular_fingerprints(fila):
    idx, smiles = fila
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return idx, None, None
        
        # MACCS Keys (167 bits)
        maccs = list(MACCSkeys.GenMACCSKeys(mol))
        
        # Morgan Fingerprint / ECFP4 (Radio 2, 1024 bits)
        morgan = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024))
        
        return idx, maccs, morgan
    except:
        return idx, None, None

# Extraer datos y configurar pool
datos_entrada = list(zip(df.index, df['canonical_smiles']))
NUCLEOS = max(1, cpu_count() - 4)
resultados_maccs = {}
resultados_morgan = {}

print(f"Procesando huellas moleculares usando {NUCLEOS} núcleos...")
with Pool(processes=NUCLEOS) as pool:
    for idx, maccs, morgan in tqdm(pool.imap_unordered(calcular_fingerprints, datos_entrada, chunksize=200), total=len(datos_entrada)):
        if maccs is not None and morgan is not None:
            resultados_maccs[idx] = maccs
            resultados_morgan[idx] = morgan

# Ensamblar los DataFrames de Fingerprints
df_maccs = pd.DataFrame.from_dict(resultados_maccs, orient='index', columns=[f'MACCS_{i}' for i in range(167)])
df_morgan = pd.DataFrame.from_dict(resultados_morgan, orient='index', columns=[f'ECFP4_{i}' for i in range(1024)])

# Unir con el DataFrame principal limpiando los fallos
df_quimico = df.loc[df_maccs.index].copy()
df_quimico = pd.concat([df_quimico, df_maccs, df_morgan], axis=1)

print(f"Fingerprints calculados exitosamente para {len(df_quimico)} moléculas.")


BLOQUE 2: Generación de Fingerprints (MACCS + ECFP4)
Procesando huellas moleculares usando 44 núcleos...


  0%|          | 0/7362 [00:00<?, ?it/s]

Fingerprints calculados exitosamente para 7362 moléculas.


In [14]:
import deepchem as dc
import pickle

print("\nBLOQUE 3: Generación de Grafos Moleculares (DeepChem)")

# Usamos el ConvMolFeaturizer que transforma SMILES en un objeto Grafo (Nodos y Aristas)
featurizer = dc.feat.ConvMolFeaturizer()

# Extraemos los SMILES válidos que sobrevivieron al bloque 2
smiles_validos = df_quimico['canonical_smiles'].tolist()

print("Featurizando moléculas a grafos...")
# Esto genera una lista de objetos ConvMol (Grafos)
grafos = featurizer.featurize(smiles_validos)

# Verificamos fallos de DeepChem y filtramos
indices_validos_grafos = [i for i, grafo in enumerate(grafos) if grafo is not None]
grafos_limpios = [grafos[i] for i in indices_validos_grafos]

# Filtramos también nuestro DataFrame químico para que coincida exactamente con los grafos
df_quimico = df_quimico.iloc[indices_validos_grafos].reset_index(drop=True)

# Exportamos los grafos a disco (Las GNN leerán este archivo directamente)
with open('grafos_tcruzi_gnn.pkl', 'wb') as f:
    pickle.dump(grafos_limpios, f)

print(f"✅ {len(grafos_limpios)} Grafos exportados como 'grafos_tcruzi_gnn.pkl'")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/home/pedro/bin/miniforge-pypy3/envs/QSAR/lib/python3.10/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/home/pedro/bin/miniforge-pypy3/envs/QSAR/lib/python3.10/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module 


BLOQUE 3: Generación de Grafos Moleculares (DeepChem)
Featurizando moléculas a grafos...
✅ 7362 Grafos exportados como 'grafos_tcruzi_gnn.pkl'


In [15]:
from sklearn.feature_selection import VarianceThreshold

print("\nBLOQUE 4: Selección de Características (Feature Selection)")

# Seleccionamos solo las columnas de Fingerprints
columnas_fp = [c for c in df_quimico.columns if c.startswith('MACCS_') or c.startswith('ECFP4_')]
X_fp = df_quimico[columnas_fp].astype(int) # Los fingerprints son binarios (0 o 1)

print(f"Descriptores originales: {X_fp.shape[1]}")

# 1. Filtro de Varianza (Eliminar columnas que siempre son 0 o siempre son 1)
selector_var = VarianceThreshold(threshold=0.01) # Varianza mínima del 1%
X_var = selector_var.fit_transform(X_fp)
columnas_con_varianza = X_fp.columns[selector_var.get_support()]
X_var_df = pd.DataFrame(X_var, columns=columnas_con_varianza)

print(f"Descriptores tras filtro de varianza: {X_var_df.shape[1]}")

# 2. Filtro de Correlación Colineal (>0.95)
# NOTA: Calcular correlación en 1000+ columnas toma mucha RAM.
print("Calculando matriz de correlación de Pearson (esto puede tomar un minuto)...")
matriz_corr = X_var_df.corr().abs()
upper_tri = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))

columnas_a_eliminar = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
X_final_fp = X_var_df.drop(columns=columnas_a_eliminar)

print(f"Descriptores retenidos tras filtro de correlación: {X_final_fp.shape[1]}")

# Reensamblamos el DataFrame con los metadatos y los features limpios
columnas_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico', 'pchembl_value']
df_features_limpios = pd.concat([df_quimico[columnas_metadatos], X_final_fp], axis=1)


BLOQUE 4: Selección de Características (Feature Selection)
Descriptores originales: 1191
Descriptores tras filtro de varianza: 959
Calculando matriz de correlación de Pearson (esto puede tomar un minuto)...
Descriptores retenidos tras filtro de correlación: 941


In [16]:
print("\nBLOQUE 5: Construcción de la Matriz IBMTL (Aprendizaje Multitarea)")

df_ibmtl = df_features_limpios.copy()

# 1. Aplicar One-Hot Encoding al blanco terapéutico
print("Aplicando One-Hot Encoding a las dianas biológicas...")
df_ibmtl = pd.get_dummies(df_ibmtl, columns=['blanco_terapeutico'], prefix='Target')

# 2. Inyectar los Pesos de Similitud Evolutiva
print("Inyectando matriz de similitud de secuencia (AA-GSS)...")
# Creamos nuevas columnas para la similitud hacia cada blanco
for blanco_referencia in blancos:
    nombre_columna_similitud = f'Similitud_{blanco_referencia}'
    
    # Buscamos en qué columna de Target está (para recuperar el nombre original de la fila)
    def obtener_similitud(fila):
        # Averiguamos a qué blanco pertenece esta fila específica (donde el One-Hot es True)
        for b in blancos:
            col_target = f'Target_{b}'
            if col_target in fila and fila[col_target] == 1:
                # Retornamos la similitud precalculada entre el blanco de la fila y el blanco_referencia
                return matriz_similitud[b][blanco_referencia]
        return 0.0 # Por seguridad
        
    df_ibmtl[nombre_columna_similitud] = df_ibmtl.apply(obtener_similitud, axis=1)

# 3. Exportación final
nombre_archivo_ibmtl = 'matriz_ibmtl_tcruzi_mlclasico.csv'
df_ibmtl.to_csv(nombre_archivo_ibmtl, index=False)

print(f"🚀 ¡Todo listo! La matriz IBMTL fue guardada como '{nombre_archivo_ibmtl}'")
print(f"Columnas finales del modelo: {df_ibmtl.shape[1]}")


BLOQUE 5: Construcción de la Matriz IBMTL (Aprendizaje Multitarea)
Aplicando One-Hot Encoding a las dianas biológicas...
Inyectando matriz de similitud de secuencia (AA-GSS)...
🚀 ¡Todo listo! La matriz IBMTL fue guardada como 'matriz_ibmtl_tcruzi_mlclasico.csv'
Columnas finales del modelo: 953


In [18]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

print("BLOQUE 6: Modelado Base y Doble Validación Cruzada (XGBoost Regressor)")

# 1. Cargar la Matriz IBMTL 
df = pd.read_csv('matriz_ibmtl_tcruzi_mlclasico.csv')

# === CORRECCIÓN AQUÍ ===
# Ya no incluimos 'blanco_terapeutico' porque fue reemplazada por las columnas Target_...
columnas_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'pchembl_value']
X = df.drop(columns=columnas_metadatos)
y = df['pchembl_value']

print(f"Dimensiones de X (Features): {X.shape}")
print(f"Dimensiones de y (Target): {y.shape}")

# 3. Configurar la Doble Validación Cruzada (Nested CV)
cv_externo = KFold(n_splits=5, shuffle=True, random_state=42)
cv_interno = KFold(n_splits=3, shuffle=True, random_state=42)

modelo_xgb = xgb.XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42)

espacio_parametros = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

busqueda = RandomizedSearchCV(
    estimator=modelo_xgb, 
    param_distributions=espacio_parametros, 
    n_iter=15, 
    scoring='neg_mean_squared_error', 
    cv=cv_interno, 
    n_jobs=-1, 
    random_state=42,
    verbose=1
)

# 4. Ejecutar la Validación Cruzada Externa
print("\nIniciando DCV (Esto tomará unos minutos usando todos tus núcleos)...")
puntuaciones_mse = cross_val_score(busqueda, X, y, scoring='neg_mean_squared_error', cv=cv_externo, n_jobs=1)
rmse_scores = np.sqrt(-puntuaciones_mse)

print("\n📊 RESULTADOS DE LA DOBLE VALIDACIÓN CRUZADA (XGBoost):")
print(f"RMSE por pliegue: {np.round(rmse_scores, 3)}")
print(f"RMSE Promedio Global: {rmse_scores.mean():.3f} +/- {rmse_scores.std():.3f}")

# 5. Entrenar el modelo final
print("\nEntrenando modelo XGBoost final de producción...")
busqueda.fit(X, y)
modelo_final = busqueda.best_estimator_

y_pred_total = modelo_final.predict(X)
r2_global = r2_score(y, y_pred_total)
mae_global = mean_absolute_error(y, y_pred_total)

print(f"Mejores Hiperparámetros Encontrados: {busqueda.best_params_}")
print(f"R2 (Varianza explicada) del modelo final: {r2_global:.3f}")
print(f"Error Absoluto Medio (MAE): {mae_global:.3f} log unidades")

joblib.dump(modelo_final, 'modelo_mtqsar_xgboost_tcruzi.pkl')
print("\n💾 ¡Modelo XGBoost maestro guardado!")

BLOQUE 6: Modelado Base y Doble Validación Cruzada (XGBoost Regressor)
Dimensiones de X (Features): (7362, 950)
Dimensiones de y (Target): (7362,)

Iniciando DCV (Esto tomará unos minutos usando todos tus núcleos)...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Fitting 3 folds for each of 15 candidates, totalling 45 fits

📊 RESULTADOS DE LA DOBLE VALIDACIÓN CRUZADA (XGBoost):
RMSE por pliegue: [0.613 0.605 0.624 0.63  0.612]
RMSE Promedio Global: 0.617 +/- 0.009

Entrenando modelo XGBoost final de producción...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Mejores Hiperparámetros Encontrados: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 9, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
R2 (Varianza explicada) del modelo final: 0.965
Error Absoluto Medio (MAE): 0.156 

In [19]:
pwd

'/home/pedro/bin'